##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Agents API: Build managed agents with the Interactions API

<a class="tfo-notebook-buttons" target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_managed_agents.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The [Interactions API](https://ai.google.dev/gemini-api/docs/interactions) provides a unified interface for working with Gemini models and agents. The [Getting Started notebook](./Get_started_interactions_api.ipynb) covers how to use it with standard Gemini **models** for text generation, multi-turn conversations, and tool use.

This notebook focuses on something different: **managed agents** with the `antigravity-preview-05-2026` agent.

### `agent=` vs `model=`

When you call the Interactions API, you choose between two modes:

| Parameter | What runs | Best for |
|-----------|-----------|----------|
| `model="gemini-..."` | A standard Gemini model | Text generation, structured output, function calling |
| `agent="antigravity-preview-05-2026"` | A **managed agent** in a sandboxed Linux environment | Autonomous tasks: code execution, web research, file management |

With `model=`, you get a stateless LLM call (see the [Getting Started notebook](./Get_started_interactions_api.ipynb)). With `agent=`, you spin up an autonomous agent that can **reason, plan, write and execute code, browse the web, and manage files** — all inside a secure sandbox, without you writing any orchestration logic.

This notebook walks you through the agent mode step by step:

1. **Simple questions** — use the agent like an LLM (it works, but it's overkill!)
2. **Multi-turn conversations** — persistent sandbox = built-in memory
3. **Using tools** — code execution, web search, file operations
4. **Loading data into the sandbox** — inject files before the agent starts
5. **Creating reusable custom agents** — bundle instructions, skills, and environment

<a name="setup"></a>
## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai). It's recommended to always use the latest version.

In [1]:
%pip install -U -q "google-genai>=2.9.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 9.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.1 which is incompatible.


In [1]:
pip install --upgrade google-auth==2.49.0

### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key or you aren't sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [5]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

With the new SDK, now you only need to initialize a client with you API key.

In [6]:
import uuid
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=GEMINI_API_KEY)

# The default managed agent.
AGENT = "antigravity-preview-05-2026"

# Generate a unique suffix for this notebook session to prevent agent ID conflicts
UNIQUE_SUFFIX = uuid.uuid4().hex[:8]

print("Client ready!")

Client ready!


## 1. Simple questions — the agent as an LLM

The simplest way to use a managed agent is to ask it a question, just like you'd call a standard Gemini model. Pass `agent="antigravity-preview-05-2026"` and `environment="remote"` to create a fresh Linux sandbox for the agent.

This works, but it's a bit like driving a Formula 1 car to the grocery store — the agent has code execution, web search, and file management capabilities that are all sitting idle for a simple factual question.

In [7]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What is the capital of France?",
    environment="remote",
)

Markdown(interaction.output_text)

The capital of France is **Paris**.

In [8]:
# The response also includes metadata about the agent's sandbox.
print(f"Status:         {interaction.status}")
print(f"Interaction ID: {interaction.id}")
print(f"Environment ID: {interaction.environment_id}")

Status:         completed
Interaction ID: v1_ChYtczY0YXAzMkROZmxqTWNQNWVUb2VREhYtczY0YXAzMkROZmxqTWNQNWVUb2VR
Environment ID: 8a2ff322f27f58c1d4cba608dfbd1cf0


Notice the `environment_id` in the response. That's the agent's persistent Linux sandbox. Even for this simple question, a full container was provisioned. Let's make use of that persistence next.

## 2. Multi-turn conversations

Since each agent runs in a persistent sandbox, you can **continue where you left off** by reusing the `environment_id` and linking turns with `previous_interaction_id`.

This is fundamentally different from stateless `model=` calls. The agent has a true *persistent environment* — files it creates stick around, packages it installs remain available, and conversation context is preserved.

In [9]:
# Turn 1: Introduce yourself.
turn1 = client.interactions.create(
    agent=AGENT,
    input="Hi! My name is Alice and I'm a software engineer. Remember that in a knowledge.md doc.",
    environment= "remote",
)

Markdown(f"**Turn 1:** {turn1.output_text}")

**Turn 1:** Nice to meet you, Alice! I have saved your information in `knowledge.md`. Let me know if there's anything else you'd like me to add or assist you with!

In [10]:
# Turn 2: Ask whether the agent remembers.
# Pass environment_id and previous_interaction_id to continue the conversation.
turn2 = client.interactions.create(
    agent=AGENT,
    input="what's my name and what do I do?",
    environment= turn1.environment_id,
)

Markdown(f"**Turn 2:** {turn2.output_text}")

**Turn 2:** According to `knowledge.md`:

- **Name:** Alice
- **Occupation:** Software Engineer

The agent remembered across turns because you passed `environment` with the previous environment ID — same sandbox, which means same files.

You could have achieved the same result using `previous_interaction_id` to keep the history of the previous conversation, but that would not have showcased the environement specificities.

This is how you build stateful, multi-turn workflows. See the [Getting Started notebook](./Get_started_interactions_api.ipynb) for `model=`-based multi-turn using `previous_interaction_id` alone (without environments).

## 3. Using tools — where the agent shines

This is where managed agents go beyond a standard chat model. The antigravity-preview-05-2026 agent has **built-in tools** it uses autonomously — you don't declare them, just describe your goal and the agent figures out what to use.

| Tool | Description |
|------|-------------|
| `bash` | Execute shell commands in the sandbox |
| `google_search` | Search the web for current information |
| `url_context` | Fetch and extract text from URLs |
| `write_file` | Create or overwrite files in the sandbox |
| `read_file` | Read file contents from the sandbox |
| `list_files` | List directory contents |
| `delete_file` | Remove files from the sandbox |

For the standard `model=`-based tools (Google Search grounding, code execution, function calling), see the [Getting Started notebook](./Get_started_interactions_api.ipynb) and the dedicated tool notebooks:
- [Code Execution](./Code_Execution.ipynb)
- [Search Grounding](./Search_Grounding.ipynb)
- [Function Calling](./Function_calling.ipynb)

### Code execution

Ask a computational question and the agent will write code, run it in its sandbox, and return the verified result.

In [30]:
m_tpei_config_data = """# M-TPEI v6.0 Configuration
VERSION=6.0
MODULES=data_preprocessing,model_training,evaluation
DEBUG_MODE=True
"""

interaction = client.interactions.create(
    agent=AGENT,
    input="`m_tpei_config.ini` 파일을 읽고, DEBUG_MODE가 True인지 확인한 다음, 전체 내용을 나에게 말해줘.",
    environment={
        "type": "remote",
        "sources": [
            {
                "type": "inline",
                "content": m_tpei_config_data,
                "target": "/workspace/m_tpei_config.ini",
            }
        ],
    },
)

Markdown(interaction.output_text)

`m_tpei_config.ini` 파일을 확인한 결과, **`DEBUG_MODE`는 `True`**로 설정되어 있습니다.

파일의 전체 내용은 다음과 같습니다:

```ini
# M-TPEI v6.0 Configuration
VERSION=6.0
MODULES=data_preprocessing,model_training,evaluation
DEBUG_MODE=True
```

인라인 방식은 내용이 직접 코드에 포함되기 때문에 작은 파일에 적합합니다.

**2. Google Cloud Storage (GCS)에서 파일 로드하기 (예: M-TPEI v6.0 학습 데이터셋)**

이 방법을 사용하려면 먼저 Google Cloud 프로젝트가 설정되어 있어야 하고, 파일이 GCS 버킷에 업로드되어 있어야 합니다. `BQ_PROJECT`와 같은 변수를 사용하고 있다면 해당 프로젝트 ID를 `project_id`로 사용할 수 있습니다.

In [31]:
# GCS에서 파일을 로드하는 예시 (실행하려면 유효한 GCS 경로가 필요합니다)
# 'your-gcs-bucket/m-tpei-data/' 경로를 실제 파일이 있는 GCS 버킷 경로로 변경하세요.

# BQ_PROJECT 변수가 정의되어 있다면, 이를 사용할 수 있습니다.
# 그렇지 않다면, 'your-gcp-project-id'를 실제 프로젝트 ID로 대체하세요.

# try:
#     from google.colab import userdata
#     GCP_PROJECT_ID = userdata.get('GCP_PROJECT_ID')
# except:
#     GCP_PROJECT_ID = 'your-gcp-project-id' # Replace with your actual GCP Project ID

# interaction = client.interactions.create(
#     agent=AGENT,
#     input="GCS에서 로드된 'm-tpei-data.csv' 파일을 분석하고 데이터 요약을 제공해줘.",
#     environment={
#         "type": "remote",
#         "sources": [
#             {
#                 "type": "gcs",
#                 "source": "gs://your-gcs-bucket/m-tpei-data/", # 실제 GCS 버킷 경로로 변경
#                 "target": "/workspace/m-tpei-data/",
#                 "project_id": GCP_PROJECT_ID # 필요한 경우 GCP 프로젝트 ID를 지정
#             }
#         ],
#     },
# )

# Markdown(interaction.output_text)

GCS 로딩 코드는 현재 주석 처리되어 있습니다. 실행하려면 주석을 해제하고 `your-gcs-bucket/m-tpei-data/`를 실제 GCS 경로로 교체해야 합니다.

**3. GitHub 리포지토리(Repository)에서 파일 로드하기 (예: M-TPEI v6.0의 전체 코드 베이스)**

M-TPEI v6.0의 전체 코드 또는 관련 스킬들이 GitHub 리포지토리에 있다면 이 방법을 사용할 수 있습니다.

In [32]:
# GitHub 리포지토리에서 로드하는 예시 (실행하려면 유효한 GitHub 리포지토리 URL이 필요합니다)
# 'https://github.com/your-org/m-tpei-v6-repo' 경로를 실제 GitHub 리포지토리 URL로 변경하세요.

# interaction = client.interactions.create(
#     agent=AGENT,
#     input="GitHub에서 로드된 M-TPEI v6.0 리포지토리를 살펴보고 `README.md` 파일을 요약해줘.",
#     environment={
#         "type": "remote",
#         "sources": [
#             {
#                 "type": "repository",
#                 "source": "https://github.com/your-org/m-tpei-v6-repo", # 실제 GitHub 리포지토리 URL로 변경
#                 "target": "/workspace/m-tpei-v6/",
#             }
#         ],
#     },
# )

# Markdown(interaction.output_text)

GitHub 리포지토리 로딩 코드도 현재 주석 처리되어 있습니다. 실행하려면 주석을 해제하고 `https://github.com/your-org/m-tpei-v6-repo`를 실제 리포지토리 URL로 교체해야 합니다.

이러한 `sources` 필드를 사용하여 M-TPEI v6.0 작업에 필요한 모든 파일을 에이전트의 샌드박스에 로드할 수 있습니다. 에이전트는 로드된 파일들을 `/workspace` 경로 아래에서 접근할 수 있습니다. 어떤 방식이 M-TPEI v6.0에 가장 적합한가요?

In [11]:
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Write a Python script that computes the first 20 Fibonacci numbers. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

I have created and executed the Python script `fibonacci.py`.

### Script (`fibonacci.py`)

```python
def get_fibonacci(n):
    """Generate the first n Fibonacci numbers."""
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    
    fib = [0, 1]
    while len(fib) < n:
        fib.append(fib[-1] + fib[-2])
    return fib

if __name__ == "__main__":
    n = 20
    fib_series = get_fibonacci(n)
    print(f"First {n} Fibonacci numbers:")
    for idx, num in enumerate(fib_series, start=1):
        print(f"F({idx}) = {num}")
```

### Execution Output

```
First 20 Fibonacci numbers:
F(1) = 0
F(2) = 1
F(3) = 1
F(4) = 2
F(5) = 3
F(6) = 5
F(7) = 8
F(8) = 13
F(9) = 21
F(10) = 34
F(11) = 55
F(12) = 89
F(13) = 144
F(14) = 233
F(15) = 377
F(16) = 610
F(17) = 987
F(18) = 1597
F(19) = 2584
F(20) = 4181
```

### Inspecting steps — what the agent actually did

The `steps` field in the response shows the agent's reasoning chain: its thoughts, tool calls, tool results, and final output. This is useful for debugging and understanding the agent's behavior.

In [12]:
# Inspect the steps from the Fibonacci interaction above.
for i, step in enumerate(interaction.steps):
    step_type = step.type
    print(f"--- Step {i} [{step_type}] ---")

    # Tool call steps show which tool was invoked and with what arguments.
    if hasattr(step, "name") and step.name:
        print(f"  Tool: {step.name}")
        if hasattr(step, "arguments"):
            args_str = str(step.arguments)[:300]
            print(f"  Args: {args_str}")

    # Content steps contain the agent's text output.
    if hasattr(step, "content") and step.content:
        for c in step.content:
            if hasattr(c, "text"):
                print(f"  Text: {c.text[:300]}")
    print()

--- Step 0 [thought] ---

--- Step 1 [function_call] ---
  Tool: write_file
  Args: {'path': 'fibonacci.py', 'toolSummary': 'Python script creation', 'content': 'def get_fibonacci(n):\n    """Generate the first n Fibonacci numbers."""\n    if n <= 0:\n        return []\n    elif n == 1:\n        return [0]\n    \n    fib = [0, 1]\n    while len(fib) < n:\n        fib.append(fib[-1]

--- Step 2 [function_result] ---
  Tool: write_file

--- Step 3 [code_execution_call] ---

--- Step 4 [code_execution_result] ---

--- Step 5 [model_output] ---
  Text: I have created and executed the Python script `fibonacci.py`.

### Script (`fibonacci.py`)

```python
def get_fibonacci(n):
    """Generate the first n Fibonacci numbers."""
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    
    fib = [0, 1]
    while len(fib) < n:
        fib



### Web search

The agent can search the web autonomously when it needs up-to-date information.

In [13]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What were the top 3 news stories about Google this week? Summarize them briefly.",
    environment="remote",
)

Markdown(interaction.output_text)

Here are the top three news stories about Google this week (September 20–27, 2026):

---

### 1. Google Fined €403 Million ($463 Million) by EU Privacy Regulators
* **Summary:** Ireland’s Data Protection Commission (DPC)—acting as Google's lead regulator across the European Union—issued a €403 million fine against Google for violating the EU's General Data Protection Regulation (GDPR) [1, 2]. 
* **Key Details:** The multi-year investigation concluded that between 2018 and 2020, Google failed to process users' location data lawfully, fairly, and transparently across three features: *Web & App Activity*, *Location History*, and Android’s *Location Accuracy*. Regulators noted that users were often unaware their location history was retained and used for advertising profiles. Google was ordered to bring its data processing procedures into full compliance within six months; the company responded that the probe focused on historical practices that have since been overhauled [1, 2].

---

### 2. UK Antitrust Watchdog Proposes Opening Android & Chrome to AI Search Rivals
* **Summary:** The UK’s Competition and Markets Authority (CMA) published updated, strengthened competition proposals that would require Google to offer users choice screens not only for traditional search engines but also for rival AI assistants [3, 4].
* **Key Details:** Under the draft conduct rules, Google would be mandated to present setup screens on Android devices and Chrome (desktop and mobile) where users pick their default search provider, followed by an annual prompt. Crucially, the CMA expanded eligibility beyond standard search engines to include AI platforms such as ChatGPT and Perplexity, while requiring all eligible search services to provide clear citations and attribution to original web publishers [3, 4].

---

### 3. Google Announces First Orbital Space Test for "Project Suncatcher" AI Chips
* **Summary:** Google revealed that its research moonshot to operate AI data centers in orbit, **Project Suncatcher**, will launch its first prototype satellite into space aboard SpaceX’s Transporter-18 rideshare mission [5, 6].
* **Key Details:** Developed in collaboration with satellite company Planet, the experimental satellite carries four Google Trillium Tensor Processing Units (TPUs). The mission aims to evaluate how Google’s proprietary machine learning hardware withstands the intense physical stresses of launch, cosmic radiation, and vacuum thermal cooling in low Earth orbit. Google hopes to use near-constant solar energy in space to scale future AI compute beyond terrestrial energy constraints [5, 6].

---

### References
* [1] [Google Fined $463 Million Over E.U. Location Data Violations - Quartz](https://qz.com/google-fined-463-million-eu-location-data-gdpr-092126)
* [2] [European Union Fines Google $590 Million for Data Location Breach - The Straits Times](https://www.straitstimes.com/world/europe/european-union-fines-google-590-million-for-data-location-breach)
* [3] [CMA Strengthens Proposals Allowing People Choice Over Their Search Service - GOV.UK](https://www.gov.uk/government/news/cma-strengthens-proposals-allowing-people-choice-over-their-search-service)
* [4] [UK Competition Watchdog Wants to Give Android and Chrome Users More Search Engine Choices - Engadget](https://www.engadget.com/2266226/uk-competition-watchdog-wants-to-give-android-and-chrome-users-more-search-engine-choices/)
* [5] [Behind Project Suncatcher, Our Moonshot to Put AI in Space - Google Blog](https://blog.google/innovation-and-ai/models-and-research/google-research/google-project-suncatcher-facts/)
* [6] [Project Suncatcher: Google's Proven AI Chips Face Space Risk - Progressive Robot](https://www.progressiverobot.com/2026/09/24/project-suncatcher-google-first-test-ai-chips-in-space/)

### File operations

The agent can create, read, and manage files in its sandbox. Files persist within the environment across turns.

In [14]:
# Ask the agent to create a file, run it, and show results.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a Python file called 'analysis.py' that generates 50 random numbers, "
        "computes mean, median, and standard deviation, then prints the results. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

I have created `analysis.py` to generate 50 random numbers and compute their summary statistics (mean, median, and sample standard deviation) using Python's built-in `random` and `statistics` modules.

### Code (`analysis.py`)

```python
import random
import statistics

def main():
    # Generate 50 random floating-point numbers between 1 and 100
    numbers = [random.uniform(1.0, 100.0) for _ in range(50)]

    # Compute statistics
    mean_val = statistics.mean(numbers)
    median_val = statistics.median(numbers)
    stdev_val = statistics.stdev(numbers)

    print("Generated 50 random numbers:")
    formatted_numbers = [round(n, 2) for n in numbers]
    print(formatted_numbers)
    print("\nSummary Statistics:")
    print(f"Count:              {len(numbers)}")
    print(f"Mean:               {mean_val:.4f}")
    print(f"Median:             {median_val:.4f}")
    print(f"Standard Deviation: {stdev_val:.4f}")

if __name__ == "__main__":
    main()
```

---

### Execution Output

```text
Generated 50 random numbers:
[78.17, 96.8, 71.2, 29.25, 89.57, 54.22, 26.72, 57.81, 76.53, 29.5, 63.38, 1.64, 16.47, 10.23, 20.56, 31.93, 97.83, 60.27, 42.59, 7.23, 48.27, 60.96, 6.05, 81.4, 94.63, 40.89, 54.14, 64.89, 92.19, 49.11, 61.36, 36.02, 57.22, 65.54, 42.61, 2.81, 83.08, 87.01, 63.61, 47.67, 50.12, 63.5, 93.99, 23.0, 15.34, 90.64, 39.89, 73.76, 44.1, 35.17]

Summary Statistics:
Count:              50
Mean:               52.6181
Median:             54.1791
Standard Deviation: 27.4185
```

## 4. Loading data into the agent's sandbox

You can inject files into the agent's environment **before it starts** using `sources`. This is how you provide data, configuration, or code for the agent to work with.

| Source type | Description | Best for |
|------------|-------------|----------|
| `inline` | Embed content directly (max 75 KB) | Config files, small scripts |
| `gcs` | Load from Google Cloud Storage | Large datasets |
| `repository` | Load from GitHub | Code repositories |

In [15]:
# Inject a CSV file inline and ask the agent to analyze it.
csv_data = """name,age,city,score
              Alice,28,Paris,92
              Bob,35,London,87
              Charlie,42,Berlin,95
              Diana,31,Tokyo,88
              Eve,26,Sydney,91"""

interaction = client.interactions.create(
    agent=AGENT,
    input="Read the file data.csv, analyze it, and tell me who scored the highest.",
    environment={
        "type": "remote",
        "sources": [
            {
                "type": "inline",
                "content": csv_data,
                "target": "/workspace/data.csv",
            }
        ],
    },
)

Markdown(interaction.output_text)

Based on the contents of `data.csv`:

| Name | Age | City | Score |
| :--- | :--- | :--- | :--- |
| Alice | 28 | Paris | 92 |
| Bob | 35 | London | 87 |
| **Charlie** | **42** | **Berlin** | **95** |
| Diana | 31 | Tokyo | 88 |
| Eve | 26 | Sydney | 91 |

**Charlie** scored the highest with a score of **95**.

You can also load from other sources:

```python
# From Google Cloud Storage
{"type": "gcs", "source": "gs://my-bucket/data/", "target": "/workspace/data/"}

# From a GitHub repository
{"type": "repository", "source": "https://github.com/user/repo", "target": "/workspace/repo/"}
```

You can combine multiple sources in a single request — the agent will have access to all of them at startup.

**Pro tip:** You can use that to add skills to you agent, as you'll see next.

## 5. Creating reusable custom agents

So far, every interaction has used the base `antigravity-preview-05-2026` agent with inline instructions. Once you've found a setup that works well, you can **persist it into a named custom agent** that bundles:

- **Instructions** — system prompt that defines the agent's behavior
- **Environment** — pre-configured sandbox with files and sources
- **Skills** — `SKILL.md` files that teach the agent specialized capabilities

This is the recommended workflow:
1. **Prototype** with `agent="antigravity-preview-05-2026"` — iterate on instructions, sources, and prompts
2. **Create** a named agent via the `/agents` endpoint
3. **Invoke** your agent by name from any client

### Creating a custom agent

In [16]:
# Create a custom data analysis agent using the SDK.
my_agent = client.agents.create(
    id=f"my-data-analyst-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a data analysis assistant. "
        "Always write Python code using pandas to answer questions. "
        "Show your code and output clearly. "
        "When creating visualizations, save them as PNG files."
    ),
    base_environment={
        "type": "remote",
    },
)

print(f"✓ Agent created: {my_agent.id}")

/tmp/ipykernel_12876/1261841246.py:2: UserWarning: Agents usage is experimental and may change in future versions.
  my_agent = client.agents.create(


✓ Agent created: my-data-analyst-6c7ea8fe


### Using a custom agent

Once created, invoke your agent by name. It will follow its instructions automatically.

In [17]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-data-analyst-{UNIQUE_SUFFIX}",
    input=(
        "Generate a sample dataset of 100 sales records with columns: "
        "product, region, revenue, quantity. "
        "Find the top 5 products by total revenue and show the analysis."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

Here is the complete data analysis using Python and **pandas** to generate the sample dataset, identify the top 5 products by total revenue, and examine their performance.

---

### 1. Python Code

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

# Define product and region categories
products = [
    "Laptop", "Smartphone", "Tablet", "Monitor", 
    "Headphones", "Keyboard", "Smartwatch", "Camera"
]
regions = ["North", "South", "East", "West"]

# Generate 100 sales records
n_records = 100

base_prices = {
    "Laptop": 1000,
    "Smartphone": 700,
    "Tablet": 450,
    "Monitor": 300,
    "Headphones": 150,
    "Keyboard": 80,
    "Smartwatch": 250,
    "Camera": 600,
}

sampled_products = np.random.choice(products, size=n_records)
sampled_regions = np.random.choice(regions, size=n_records)
quantities = np.random.randint(1, 15, size=n_records)

# Calculate revenue with variation (+/- 10%)
price_variations = np.random.uniform(0.9, 1.1, size=n_records)
revenues = np.round([
    quantities[i] * base_prices[sampled_products[i]] * price_variations[i]
    for i in range(n_records)
], 2)

df = pd.DataFrame({
    "product": sampled_products,
    "region": sampled_regions,
    "revenue": revenues,
    "quantity": quantities
})

# Save dataset
df.to_csv("sales_data.csv", index=False)

# Aggregate metrics by product
product_summary = df.groupby("product").agg(
    total_revenue=("revenue", "sum"),
    total_quantity=("quantity", "sum"),
    transaction_count=("revenue", "count"),
    avg_order_value=("revenue", "mean")
).reset_index()

# Calculate revenue share percentage
total_revenue_all = df["revenue"].sum()
product_summary["revenue_pct"] = (product_summary["total_revenue"] / total_revenue_all) * 100

# Top 5 products by revenue
top_5_products = product_summary.sort_values(by="total_revenue", ascending=False).head(5).reset_index(drop=True)

# Regional breakdown for top 5 products
top_5_names = top_5_products["product"].tolist()
top_5_df = df[df["product"].isin(top_5_names)]
regional_breakdown = top_5_df.pivot_table(
    index="product",
    columns="region",
    values="revenue",
    aggfunc="sum",
    fill_value=0
).loc[top_5_names]

# Create visualization
plt.figure(figsize=(10, 6))
bars = plt.bar(
    top_5_products["product"], 
    top_5_products["total_revenue"] / 1000, 
    color="#2b5c8f", 
    edgecolor="black", 
    alpha=0.85
)

plt.title("Top 5 Products by Total Revenue", fontsize=15, pad=15, fontweight="bold")
plt.xlabel("Product", fontsize=12, labelpad=10)
plt.ylabel("Total Revenue (in Thousands USD)", fontsize=12, labelpad=10)
plt.grid(axis="y", linestyle="--", alpha=0.5)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2.0, yval + 1, f"${yval:.1f}k", ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("top_5_products_revenue.png", dpi=300)
plt.close()
```

---

### 2. Output and Results

#### Dataset Overview (First 5 Rows)
```
      product region  revenue  quantity
0  Smartwatch   East  1810.25         7
1     Monitor  South   315.70         1
2  Headphones  South   568.52         4
3  Smartwatch   West  3398.34        13
4      Tablet  South  1752.40         4
```

#### Top 5 Products by Total Revenue
| Rank | Product | Total Revenue | Units Sold | Transactions | Avg Order Value | Revenue Share |
|---|---|---|---|---|---|---|
| 1 | **Smartphone** | $86,820.98 | 118 | 13 | $6,678.54 | 27.65% |
| 2 | **Laptop** | $57,291.34 | 56 | 8 | $7,161.42 | 18.24% |
| 3 | **Camera** | $49,094.34 | 82 | 11 | $4,463.12 | 15.63% |
| 4 | **Monitor** | $40,748.76 | 137 | 19 | $2,144.67 | 12.98% |
| 5 | **Tablet** | $32,381.79 | 74 | 11 | $2,943.80 | 10.31% |

#### Regional Revenue Distribution for Top 5 Products
| Product | East | North | South | West | Total |
|---|---|---|---|---|---|
| **Smartphone** | $9,710.25 | $27,757.30 | $24,846.46 | $24,506.97 | $86,820.98 |
| **Laptop** | $26,533.28 | $23,394.14 | $944.85 | $6,419.07 | $57,291.34 |
| **Camera** | $15,839.06 | $0.00 | $12,025.35 | $21,229.93 | $49,094.34 |
| **Monitor** | $13,577.03 | $10,467.71 | $4,676.89 | $12,027.13 | $40,748.76 |
| **Tablet** | $5,211.43 | $12,584.03 | $7,708.74 | $6,877.59 | $32,381.79 |

---

### 3. Key Insights & Analysis

1. **Revenue Concentration**:
   - The top 5 products account for **$266,337.21 (84.8%)** of total sales revenue across all 100 transactions ($314,012.55 total).
2. **Top Performer (Smartphone)**:
   - **Smartphones** are the top revenue generator ($86,820.98), contributing over a quarter (27.65%) of overall sales with strong sales across North, South, and West regions.
3. **High-Value vs. High-Volume Drivers**:
   - **Laptops** have the highest average order value ($7,161.42 per transaction), generating $57,291.34 across just 8 transactions.
   - **Monitors** represent the highest volume (137 units sold across 19 orders), but rank 4th in revenue due to lower unit prices.
4. **Regional Observations**:
   - **North** and **West** regions lead across most categories.
   - **Camera** sales showed strong performance in the West ($21,229.93) and East ($15,839.06), but recorded no sales in the North.

---

### 4. Generated Artifacts
- `sales_data.csv` – CSV file containing the 100 sales records.
- `top_5_products_revenue.png` – High-resolution bar chart showing the total revenue for the top 5 products.

### Creating an agent with pre-loaded data

You can define the agent's environment with sources, so data is ready before the agent starts:

In [18]:
# Create an agent with GCS sources pre-loaded using the SDK.
my_slides_agent = client.agents.create(
    id=f"my-gemini-api-agent-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a software engineer speciliazed in the Gemini API. "
        "Use the skills available in /.agents/skills/ to create amazing apps."
    ),
    base_environment={
        "type": "remote",
        "sources": [
            {
                "type": "repository",
                "source": "https://github.com/google-gemini/gemini-skills",
                "target": "/.agents/skills",
            }
        ],
    },
)

print(f"✓ Agent created: {my_slides_agent.id}")

✓ Agent created: my-gemini-api-agent-6c7ea8fe


In [19]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-gemini-api-agent-{UNIQUE_SUFFIX}",
    input="Tell me what you can do with your skills?",
    environment="remote",
)

Markdown(interaction.output_text)

With the specialized skills available in the environment, I can architect and build full-stack, multimodal, and real-time AI applications powered by the Gemini platform using modern SDKs (`google-genai` for Python and `@google/genai` for JavaScript/TypeScript).

Here is a breakdown of what I can do across the three specialized skill sets:

---

### 1. General & Multimodal App Development (`gemini-api-dev`)
Leveraging the **Interactions API** and the latest model generations:

- **Next-Gen Text & Multimodal Reasoning**:
  - Integrate flagship models like `gemini-3.8-flash` (balanced agentic & multimodal execution) and `gemini-3.1-pro-preview` (complex reasoning, coding, and mathematical analysis).
  - Deploy high-throughput, cost-optimized pipelines with `gemini-3.5-flash-lite` and open-weights Gemma models (`gemma-4-31b-it`, `gemma-4-26b-a4b-it`).
  - Multimodal embedding indexing for text, images, video, and audio using `gemini-embedding-2`.

- **Generative Media (Image & Voice)**:
  - **Image generation & editing**: Generate and edit images with the Nano Banana suite (`gemini-3-pro-image`, `gemini-3.1-flash-image`, `gemini-3.1-flash-lite-image`).
  - **Expressive Text-to-Speech (TTS)**: Synthesize multi-speaker dialogue, voice design personas, and replicated voices using `gemini-3.8-flash-tts` and `gemini-3.8-flash-lite-tts`.
  - **Audio Transcription**: Process speech with `gemini-3.5-transcribe` supporting both `smart` (cleaned & formatted) and `verbatim` modes.

- **Grounding, Tools & MCP Integration**:
  - Connect models to function calling, Code Execution, Google Search grounding, URL Context, Maps grounding, Computer Use, and remote MCP servers.
  - Enforce strict JSON output with structured schemas.

- **Managed & Autonomous Agents**:
  - **Antigravity Agent (`antigravity-preview-09-2026`)**: Build autonomous workflows running inside Google-hosted sandboxed Linux environments capable of running shell scripts, manipulating files, and browsing the web.
  - **Custom Agents**: Configure agents with attached remote repositories or custom tool configurations via `client.agents.create()`.
  - **Deep Research**: Execute asynchronous, exhaustive background research tasks using `deep-research-preview-04-2026` and `deep-research-max-preview-04-2026`.

- **Stateful Sessions & Streaming**:
  - Maintain conversational context across multi-turn sessions using `previous_interaction_id`.
  - Stream tokens, tool calls, and thought signatures in real time via Server-Sent Events (SSE).

---

### 2. Real-Time Streaming & Voice Agents (`gemini-live-api-dev`)
Leveraging bidirectional **WebSocket** connections for sub-second, human-like voice and video streaming:

- **Low-Latency Voice & Video Dialogue**:
  - Build live interactive assistants using `gemini-3.8-live` with bidirectional PCM audio streaming (16kHz in / 24kHz out) and real-time camera/screen frame streaming (JPEG).
  - Manage real-time interruptions, automatic server Voice Activity Detection (VAD), push-to-talk, and client-side Hybrid VAD (`audio_stream_end`).

- **Live Background Reasoning (Extended Thinking)**:
  - Use `gemini-3.8-live-extended-thinking` to perform multi-step planning and non-blocking asynchronous function calls (`behavior: "NON_BLOCKING"`).
  - The model delivers spoken conversational fillers (e.g., *"Let me look that up for you..."*) while executing background tools, tracked via `interaction_status` (`IN_PROGRESS` vs `IDLE`).

- **Real-Time Translation & Transcription**:
  - **Live Speech-to-Speech Translation**: Stream voice translation across 70+ languages with `gemini-3.5-live-translate-preview`.
  - **Live Speech-to-Text**: Stream real-time transcription with interim hypotheses and finalized text using `gemini-3.5-transcribe-live`.

- **Production Session Management**:
  - Issue short-lived **ephemeral tokens** for secure browser and mobile client authentication without exposing API keys.
  - Implement context window compression and session resumption to support long-running sessions.
  - Inject runtime context mid-stream using `send_client_content`.

---

### 3. Generative Video Creation & Editing (`gemini-omni-flash-api`)
Leveraging `gemini-omni-1.1-flash` via the Interactions and Files APIs:

- **Text-to-Video & Camera Control**:
  - Generate videos from natural language prompts in resolutions up to **4K** (`360p`, `720p`, `1080p`, `4k`) in landscape (`16:9`) or portrait (`9:16`) aspect ratios with synchronized audio.
- **First-Frame & Frame Interpolation**:
  - Animate static images from a starting frame (`<FIRST_FRAME>`).
  - Generate seamless transitions or infinite looping clips between two images (`<FIRST_FRAME>` and `<LAST_FRAME>`).
- **Video Extension & Continuity**:
  - Extend existing generated or uploaded clips by up to 10 seconds per turn, up to a total length of 40 seconds, keeping characters, motion, and audio coherent.
- **Video Editing & Sound Regeneration**:
  - Apply stylistic modifications (e.g., anime transformation, lighting changes) or inpainting/outpainting.
  - Choose between preserving original audio or stripping audio with `ffmpeg` to force full AI sound regeneration.
- **Multimodal Video & Image References**:
  - Use reference images (`<IMAGE_REF_N>`) or reference videos (`<VIDEO_REF_N>`) to anchor character appearance, props, or motion dynamics.
- **Batch & Automated Pipelines**:
  - Run parallel video generation jobs across multiple concurrent prompts or JSON configurations.

---

### Example Apps We Can Build

1. **Real-time Voice AI Companion / Customer Support Bot**: Runs on `gemini-3.8-live`, streams microphone audio and camera feeds over WebSockets, calls internal APIs asynchronously, and responds with low-latency spoken audio.
2. **Deep Autonomous Research Analyst**: An Antigravity or Deep Research pipeline that takes an objective, browses the web, writes and executes code to parse data, and compiles exhaustive markdown reports.
3. **Generative Video Studio**: A pipeline that takes storyboards (images), generates video transitions between keyframes, extends shots up to 40 seconds, applies style changes, and outputs 1080p/4K clips with audio.
4. **Multilingual Live Interpreter**: A real-time audio translation app utilizing `gemini-3.5-live-translate-preview` for bidirectional spoken language translation.

Tell me what you'd like to build, and we can get started right away!

### Forking from an existing environment

If you've already set up a sandbox you like (installed packages, created files, etc.), you can fork it into a new agent using the `environment_id` from a previous interaction:

```python
my_forked_agent = client.agents.create(
    id="my-forked-agent",
    base_agent=AGENT,
    system_instruction="Your custom instructions here.",
    base_environment={"env_id": "YOUR_ENVIRONMENT_ID"},
)
```

This captures the exact state of that sandbox — all installed packages, files, and configuration.

In [22]:
my_forked_agent = client.agents.create(
    id="my-forked-agent",
    base_agent=AGENT, # Changed to use the default AGENT for forking an environment
    system_instruction="I want all your apps to use the Live API",
    base_environment={"env_id": interaction.environment_id},
)
print(f"✓ Agent forked: {my_forked_agent.id}")

✓ Agent forked: my-forked-agent


### Managing agents (CRUD)

The `/agents` endpoint supports full lifecycle management:

In [24]:
# List all your agents.
print("Your agents:")
for agent in client.agents.list().agents:
    print(f"- {agent.id}")

# Get a specific agent's details.
agent = client.agents.get(id=f"my-data-analyst-{UNIQUE_SUFFIX}")
print(f"\nAgent details for {agent.id}:")
print(f"Base agent: {agent.base_agent}")
print(f"System instruction: {agent.system_instruction}")

Your agents:
- my-data-analyst-6c7ea8fe
- my-forked-agent
- my-gemini-api-agent-6c7ea8fe

Agent details for my-data-analyst-6c7ea8fe:
Base agent: antigravity-preview-05-2026
System instruction: You are a data analysis assistant. Always write Python code using pandas to answer questions. Show your code and output clearly. When creating visualizations, save them as PNG files.


In [25]:
# Clean up: delete the agents you created.
for agent_name in ["my-data-analyst", "my-forked-agent", "my-gemini-api-agent"]:
    try:
        client.agents.delete(id=agent_name)
        print(f"✓ Deleted {agent_name}")
    except Exception as e:
        print(f"  Failed to delete {agent_name}: {e}")

✓ Deleted my-data-analyst
✓ Deleted my-forked-agent
✓ Deleted my-gemini-api-agent


### Agent directory structure

Behind the API, an agent is defined by a simple set of files. This is what gets deployed when you create one:

```
my-agent/
├── agent.yaml       # Configuration: base agent, tools, environment
├── AGENTS.md        # System instructions (loaded automatically)
├── skills/          # Custom SKILL.md files that extend capabilities
└── workspace/       # Files seeded into the remote sandbox at startup
```

- **`agent.yaml`** maps directly to the `/agents` API resource
- **`AGENTS.md`** provides system instructions — automatically loaded by the harness
- **`skills/`** contains specialized `SKILL.md` files the agent discovers and uses
- **`workspace/`** files are injected into the sandbox at startup

This file-based structure makes agents easy to version-control, share, and iterate on. Check the [documentation](https://ai.google.dev/gemini-api/docs/custom-agents#file-based_customization) for more details.

## 6. Streaming

For longer tasks, enable streaming with `stream=True` to get real-time updates as the agent works. Instead of waiting for the complete response, you receive a stream of **Server-Sent Events (SSE)** that let you show progress to the user.

### Event types

The stream delivers events that tell you what the agent is doing:

| Event type | Meaning | What to do |
|------------|---------|------------|
| `interaction.created` | The interaction was created | Store the `id` for later reference |
| `interaction.status_update` | Status changed (e.g., `in_progress`) | Update UI status indicator |
| `step.start` | A new step began (thinking, tool call, output) | Show a loading indicator |
| `step.delta` | Incremental content — a chunk of text, thought, or tool output | **Append to display** — this is the main content |
| `step.stop` | A step completed | Hide loading indicator |
| `interaction.completed` | The agent finished all work | Finalize the UI |

The `step.delta` events are where the content lives. Each delta has a `type` (e.g., `text`, `thought`, `function_call`, `function_result`) and content you can render incrementally.

In [26]:
# Stream a response and collect the text as it arrives.
stream = client.interactions.create(
    agent=AGENT,
    input="Write a short poem about the ocean.",
    stream=True,
    environment="remote",
)

collected_text = []

for event in stream:
    # Show the event type so you can see the lifecycle.
    if event.event_type in ("interaction.created", "step.start", "step.stop", "interaction.completed"):
        print(f"[{event.event_type}]")

    # step.delta events carry the actual content.
    elif event.event_type == "step.delta":
        delta = event.delta
        if hasattr(delta, "text") and delta.text:
            print(delta.text, end="", flush=True)
            collected_text.append(delta.text)

print(f"\n\n--- Collected {len(collected_text)} text chunks ---")

[interaction.created]
[step.start]
Boundless blue beneath the sky,  
Where restless currents softly sigh.  
Crested waves that rise and fall,  
An ancient, endless, silver call.  

In twilight depths where shadows sleep,  
The secrets of the world run deep;  
A breathing tide upon the sand,  
The ocean whispers to the land.[step.stop]
[interaction.completed]


--- Collected 4 text chunks ---


## 7. Advanced features

### Network configuration

By default, the agent's sandbox has unrestricted outbound network access. You can control this with the `network` field:
- **Allowlist specific domains** — only requests to listed domains are permitted
- **Inject credentials** — automatically add headers (API keys, tokens) to outbound requests
- **Disable network** — set `network: "disabled"` to block all outbound traffic

In [27]:
# Allow the agent to call only the Gemini API, with an auto-injected API key.
interaction = client.interactions.create(
    agent=AGENT,
    input="Use curl to call the Gemini API and list available models. Show the first 3.",
    environment={
        "type": "remote",
        "network": {
            "allowlist": [
                {
                    "domain": "generativelanguage.googleapis.com",
                    "transform": [{"x-goog-api-key": GEMINI_API_KEY}],
                },
            ]
        },
    },
)

Markdown(interaction.output_text)

Here is the `curl` command to list the models from the Gemini API and extract the first 3 models:

```bash
curl -s https://generativelanguage.googleapis.com/v1beta/models | jq '.models[:3]'
```

### Result:

```json
[
  {
    "name": "models/gemini-2.5-flash",
    "version": "001",
    "displayName": "Gemini 2.5 Flash",
    "description": "Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.",
    "inputTokenLimit": 1048576,
    "outputTokenLimit": 65536,
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
    "maxTemperature": 2,
    "thinking": true
  },
  {
    "name": "models/gemini-2.5-pro",
    "version": "2.5",
    "displayName": "Gemini 2.5 Pro",
    "description": "Stable release (June 17th, 2025) of Gemini 2.5 Pro",
    "inputTokenLimit": 1048576,
    "outputTokenLimit": 65536,
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
    "maxTemperature": 2,
    "thinking": true
  },
  {
    "name": "models/gemini-2.5-flash-preview-tts",
    "version": "gemini-2.5-flash-exp-tts-2025-05-19",
    "displayName": "Gemini 2.5 Flash Preview TTS",
    "description": "Gemini 2.5 Flash Preview TTS",
    "inputTokenLimit": 8192,
    "outputTokenLimit": 16384,
    "supportedGenerationMethods": [
      "countTokens",
      "generateContent"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
    "maxTemperature": 2
  }
]
```

### Download environment snapshots

You can download all the files the agent created or modified as a tar archive. This lets you retrieve the agent's work products — code, data, reports — from the sandbox.

In [28]:
import subprocess
import tarfile
import os

# Create an interaction where the agent produces files.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a directory called 'project' with a README.md and a hello.py script. "
        "List the files you created."
    ),
    environment="remote",
)

env_id = interaction.environment_id
print(f"Environment ID: {env_id}")

Markdown(interaction.output_text)

Environment ID: 33e0e67860565f90a76654ea9a5c525b


The directory `project` has been created along with the requested files.

### Created Files:
- `README.md`
- `hello.py`

In [29]:
# Download the environment snapshot.
download_url = (
    f"https://generativelanguage.googleapis.com/v1beta/"
    f"files/environment-{env_id}:download?alt=media"
)

result = subprocess.run(
    ["curl", "-L", "-s", "-o", "snapshot.tar",
     "-H", f"x-goog-api-key: {GEMINI_API_KEY}",
     download_url],
    capture_output=True, text=True,
)

if os.path.exists("snapshot.tar") and os.path.getsize("snapshot.tar") > 0:
    with tarfile.open("snapshot.tar") as tar:
        print("Files in snapshot:")
        for member in tar.getmembers():
            print(f"  {member.name} ({member.size} bytes)")
else:
    print("Snapshot not available (environment may have expired).")

Files in snapshot:
  . (0 bytes)
  ./project (0 bytes)
  ./project/README.md (74 bytes)
  ./project/hello.py (78 bytes)


## Next steps

You've walked through the core capabilities of managed agents:

1. ✅ **Simple Q&A** — the agent can answer questions like an LLM
2. ✅ **Multi-turn** — persistent sandbox enables stateful conversations
3. ✅ **Built-in tools** — code execution, web search, file management
4. ✅ **Data loading** — inject files via inline, GCS, or GitHub sources
5. ✅ **Custom agents** — reusable configurations with instructions, skills, and environment
6. ✅ **Streaming** — real-time updates as the agent works
7. ✅ **Advanced** — network control and environment snapshots

### Learn more

- **[Getting Started notebook](./Get_started_interactions_api.ipynb)** — the `model=`-based Interactions API for standard generation, multi-turn, and tools
- **[Managed Agents documentation](https://ai.google.dev/gemini-api/docs/eap/gemini-agents/gemini-agents)** — full reference for the agent API
- **[Code Execution](./Code_Execution.ipynb)** — model-based code execution
- **[Search Grounding](./Search_Grounding.ipynb)** — model-based web search
- **[Function Calling](./Function_calling.ipynb)** — custom function declarations